# PCN on GraphUniverse - TDL Challenge 2026 (Track 2)

Two phases, run as separate Kaggle sessions:

| PHASE | Accelerator | What it does | Approx. time |
|---|---|---|---|
| `lift` | **None (CPU)** - does not consume GPU quota | Lifts all 24 dataset variants to path complexes and caches them | ~5-6 h |
| `train` | **GPU T4/P100** | Runs one chunk = 12 runs (one task x one seed) | ~1-2.5 h (estimate) |

Run `lift` **once**, save the version, then add its output as a data
source for six `train` sessions (2 tasks x 3 seeds).

**Required settings:** Internet **On**; Persistence **Variables and Files**.

In [ ]:
# ============================== CONFIG ==============================
PHASE = "lift"            # "lift" or "train"
MODE  = "community_detection"   # train only: community_detection | triangle_counting
SEED  = 42                      # train only: 42 | 43 | 44

# Kaggle input slugs (adjust to your actual dataset names)
CODE_DIR     = "/kaggle/input/pcn-code"       # contains pcn_kaggle_code.zip
LIFTINGS_DIR = "/kaggle/input/pcn-liftings"   # output of the lift session
# ====================================================================
assert PHASE in {"lift", "train"}
print(f"PHASE={PHASE}" + (f"  MODE={MODE}  SEED={SEED}" if PHASE == "train" else ""))

## 1. Unpack the code bundle

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working"
REPO = f"{WORK}/TopoBench"

if not os.path.isdir(REPO):
    zips = glob.glob(f"{CODE_DIR}/*.zip")
    assert zips, f"no zip found in {CODE_DIR}"
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(WORK)
    print("extracted:", zips[0])
else:
    print("code already present")

assert os.path.isfile(f"{REPO}/configs/run.yaml"), "unexpected bundle layout"
print(subprocess.run(["ls", REPO], capture_output=True, text=True).stdout)

## 2. Build the environment

Uses the repository's own `uv_env_setup.sh`, which pins torch 2.3.0 and
the matching PyG extension wheels. This creates an isolated `.venv`, so
nothing depends on Kaggle's preinstalled package versions.

Takes ~10 minutes on the first run of a session.

In [ ]:
PLATFORM = "cu121" if PHASE == "train" else "cpu"
VENV_PY = f"{REPO}/.venv/bin/python"

if not os.path.isfile(VENV_PY):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    r = subprocess.run(
        ["bash", "uv_env_setup.sh", PLATFORM],
        cwd=REPO, text=True, capture_output=True,
    )
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-4000:])
        raise SystemExit("environment setup failed")
else:
    print("venv already built")

print(subprocess.run(
    [VENV_PY, "-c",
     "import torch, topobench; print('torch', torch.__version__, "
     "'cuda', torch.cuda.is_available())"],
    cwd=REPO, capture_output=True, text=True).stdout)

## 3. Stage the lifting caches (train phase only)

Copies the cached path complexes out of the read-only input mount, since
the data pipeline needs a writable dataset directory.

In [ ]:
if PHASE == "train":
    src = None
    for cand in glob.glob(f"{LIFTINGS_DIR}/**/datasets", recursive=True):
        if os.path.isdir(cand):
            src = cand
            break
    assert src, f"no datasets/ directory found under {LIFTINGS_DIR}"
    dst = f"{REPO}/datasets"
    if not os.path.isdir(dst):
        print(f"copying {src} -> {dst} (a few minutes)...")
        shutil.copytree(src, dst)
    n = sum(1 for _ in glob.iglob(f"{dst}/**/graph2path_complex_lifting",
                                  recursive=True))
    print(f"lifting caches staged: {n} (expect 24)")
else:
    print("lift phase: caches will be created here")

## 4. Run

`lift` executes the challenge's own `check_challenge_grid`, which touches
every setting of both tasks and therefore materializes all 24 lifted
datasets. `train` runs one chunk through `run_chunk.py`, writing its 12
results to `outputs/pcn_chunks/` as soon as the chunk finishes.

In [ ]:
env = {**os.environ, "WANDB_MODE": "offline", "PROJECT_ROOT": REPO,
       "HYDRA_FULL_ERROR": "1"}

if PHASE == "lift":
    runner = f"{WORK}/_lift.py"
    with open(runner, "w") as fh:
        fh.write(
            "import sys\n"
            f"sys.path[:0] = [r'{REPO}', r'{REPO}/2026_tdl_challenge']\n"
            "from utils import check_challenge_grid, resolve_project_root\n"
            f"check_challenge_grid(project_root=resolve_project_root(r'{REPO}'),"
            " model_config='cell/pcn', quiet=True)\n"
        )
    cmd = [VENV_PY, runner]
    cwd = f"{REPO}/2026_tdl_challenge"
else:
    cmd = [VENV_PY, "run_chunk.py", "--mode", MODE, "--seed", str(SEED)]
    cwd = f"{REPO}/2026_tdl_challenge"

print("running:", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, cwd=cwd, env=env, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("exit code:", proc.returncode)
assert proc.returncode == 0, "run failed - see output above"

## 5. Keep the results

`lift`: everything under `/kaggle/working` is saved with the version; add
that output as a data source for the training sessions.

`train`: download `outputs/pcn_chunks/chunk__<mode>__s<seed>.json`. Once
all six exist, `merge_chunks.py` combines them into the final
`results.json` with 72 entries.

In [ ]:
if PHASE == "train":
    for p in glob.glob(f"{REPO}/2026_tdl_challenge/outputs/pcn_chunks/*.json"):
        shutil.copy(p, WORK)
        print("saved:", os.path.basename(p), os.path.getsize(p), "bytes")
else:
    total = sum(
        os.path.getsize(os.path.join(d, f))
        for d, _, fs in os.walk(f"{REPO}/datasets") for f in fs
    )
    print(f"datasets/ total: {total/1e9:.2f} GB (Kaggle output limit is 20 GB)")